# 04 — Service Point Demo

End-to-end demonstration of the **address lookup pipeline** for a Mississauga
residential address.

Pipeline stages:
1. **Geocoding** (`Geocoder`) — civic address → WGS84 point
2. **LDC resolution** (`LDCResolver`) — which distributor serves this address?
3. **Transformer assignment** (`VoronoiAssigner` / `TransformerFinder`) — nearest secondary transformer
4. **Path tracing** (`PathTracer`) — full grid path from building to generator
5. **Visualisation** — Voronoi transformer map + grid path diagram

Demo address: **7086 Tamar Mews, Mississauga, ON**

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import geopandas as gpd
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import numpy as np
from shapely.geometry import box, LineString, Point

from src.service_point.geocoder import Geocoder
from src.service_point.ldc_resolver import LDCResolver
from src.service_point.transformer_finder import TransformerFinder
from src.service_point.voronoi_assigner import VoronoiAssigner
from src.service_point.path_builder import PathBuilder
from src.topology.path_tracer import PathTracer
from src.utils.config_loader import load_settings

cfg = load_settings()
DEMO_ADDRESS = "7086 Tamar Mews, Mississauga, ON"

print(f"Demo address: {DEMO_ADDRESS}")

## 4.1  Load Grid Graph and Supporting Data

In [ ]:
GPKG_PATH = Path(cfg["outputs"]["grid_gpkg"])

nodes_gdf = gpd.read_file(GPKG_PATH, layer="nodes")
edges_gdf = gpd.read_file(GPKG_PATH, layer="edges")

# Rebuild NetworkX DiGraph
G = nx.DiGraph()
for _, row in nodes_gdf.iterrows():
    nid = row["node_id"]
    G.add_node(nid, **{col: row[col] for col in nodes_gdf.columns if col != "node_id"})
for _, row in edges_gdf.iterrows():
    G.add_edge(
        row["from_node"], row["to_node"],
        **{col: row[col] for col in edges_gdf.columns if col not in ("from_node", "to_node")}
    )

print(f"Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

# Load secondary transformer layer for Voronoi construction
sec_transformers = nodes_gdf[
    nodes_gdf["node_type"] == "secondary_transformer"
].copy()
sec_transformers = sec_transformers.rename(columns={"node_id": "tx_id"})
sec_transformers = sec_transformers.set_geometry("geometry")

print(f"Secondary transformers: {len(sec_transformers):,}")

## 4.2  Address Geocoding

In [ ]:
# Load municipal address dataset for Mississauga (preferred over Nominatim)
MUNICIPAL_ADDR_PATH = Path(cfg["sources"]["municipal_addresses"]["mississauga"]["local_path"])

if MUNICIPAL_ADDR_PATH.exists():
    municipal_addresses = gpd.read_file(MUNICIPAL_ADDR_PATH)
    print(f"Municipal address dataset loaded: {len(municipal_addresses):,} records")
else:
    print("Municipal addresses not cached — Nominatim fallback will be used")
    municipal_addresses = None

geocoder = Geocoder(municipal_addresses=municipal_addresses)
demo_point = geocoder.geocode(DEMO_ADDRESS)

if demo_point:
    print(f"\nGeocoded: {DEMO_ADDRESS}")
    print(f"  Longitude: {demo_point.x:.6f}")
    print(f"  Latitude:  {demo_point.y:.6f}")
else:
    print("Geocoding failed — using hardcoded Tamar Mews coordinates")
    demo_point = Point(-79.6541, 43.5788)  # Tamar Mews, Mississauga
    print(f"  Using: ({demo_point.x:.6f}, {demo_point.y:.6f})")

## 4.3  LDC Resolution

In [ ]:
from src.ingestion.oeb_fetcher import OEBFetcher

oeb = OEBFetcher()
service_areas = oeb.fetch_service_areas()

ldc_resolver = LDCResolver(service_areas_gdf=service_areas)
ldc_info = ldc_resolver.resolve_point(demo_point.x, demo_point.y)

if ldc_info:
    print(f"LDC resolved for {DEMO_ADDRESS}:")
    for k, v in ldc_info.items():
        print(f"  {k:<20} {v}")
else:
    print("LDC resolution returned no result — point may be outside OEB dataset coverage")

## 4.4  Voronoi Transformer Assignment Map

In [ ]:
# Construct Voronoi regions from secondary transformers
# Clip to a 1 km window around the demo address
WINDOW_DEG = 0.012  # ~1.2 km
local_box = box(
    demo_point.x - WINDOW_DEG, demo_point.y - WINDOW_DEG,
    demo_point.x + WINDOW_DEG, demo_point.y + WINDOW_DEG,
)

local_tx = sec_transformers[
    sec_transformers.geometry.within(local_box)
].copy()
print(f"Transformers within {WINDOW_DEG:.3f}\u00b0 window: {len(local_tx)}")

if len(local_tx) >= 2:
    from src.utils.geometry import voronoi_gdf, to_lambert, to_4326
    local_tx_lam = to_lambert(local_tx)
    clip_lam = to_lambert(gpd.GeoDataFrame(geometry=[local_box], crs="EPSG:4326")).geometry.unary_union
    voronoi_regions = voronoi_gdf(local_tx_lam, clip_to=clip_lam)
    voronoi_regions = to_4326(voronoi_regions)
    print(f"Voronoi cells generated: {len(voronoi_regions)}")
else:
    print("Fewer than 2 transformers in window — skipping Voronoi construction")
    voronoi_regions = None

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
ax.set_facecolor("#f0f4f8")

# Draw Voronoi cells
if voronoi_regions is not None and len(voronoi_regions) > 0:
    voronoi_regions.plot(
        ax=ax, alpha=0.30, edgecolor="#5d6d7e", linewidth=0.8,
        facecolor="#aed6f1"
    )

# Draw local distribution lines
local_edges = edges_gdf[
    edges_gdf.geometry.intersects(local_box)
]
if len(local_edges) > 0:
    line_color_map = {
        "tx_line":           "#c0392b",
        "primary_feeder":    "#e67e22",
        "secondary_lateral": "#27ae60",
        "service_drop":      "#95a5a6",
    }
    for etype, group in local_edges.groupby("edge_type"):
        color = line_color_map.get(etype, "#7f8c8d")
        group.plot(ax=ax, color=color, linewidth=1.2, alpha=0.7)

# Draw secondary transformers
if len(local_tx) > 0:
    local_tx.plot(ax=ax, color="#27ae60", markersize=60, zorder=5,
                  marker="^", edgecolors="white", linewidth=0.8)

# Highlight demo address
ax.plot(demo_point.x, demo_point.y, "*", color="#c0392b", markersize=18,
        zorder=10, markeredgecolor="white", markeredgewidth=1.5,
        label=DEMO_ADDRESS)

# Find and highlight assigned transformer
tx_finder = TransformerFinder(G)
nearest_tx_info = tx_finder.find_nearest(demo_point, radius_m=500)
if nearest_tx_info:
    tx_node = nearest_tx_info["tx_id"]
    tx_data = G.nodes.get(tx_node, {})
    tx_geom = tx_data.get("geometry")
    if tx_geom:
        ax.plot(tx_geom.x, tx_geom.y, "s", color="#f39c12", markersize=14,
                zorder=9, markeredgecolor="white", label=f"Assigned TX: {tx_node}")
        ax.annotate(
            f"Assigned TX\n{nearest_tx_info.get('distance_m', 0):.0f} m",
            xy=(tx_geom.x, tx_geom.y),
            xytext=(tx_geom.x + 0.002, tx_geom.y + 0.001),
            fontsize=8, color="#2c3e50",
            arrowprops=dict(arrowstyle="->", color="#2c3e50", lw=0.8),
        )

ax.set_xlim(demo_point.x - WINDOW_DEG, demo_point.x + WINDOW_DEG)
ax.set_ylim(demo_point.y - WINDOW_DEG, demo_point.y + WINDOW_DEG)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(
    f"Voronoi Transformer Assignment\n{DEMO_ADDRESS}",
    fontsize=12, fontweight="bold"
)

legend_handles = [
    mlines.Line2D([0], [0], marker="*", color="#c0392b", markersize=12,
                   linestyle="None", label="Demo address"),
    mlines.Line2D([0], [0], marker="^", color="#27ae60", markersize=10,
                   linestyle="None", label="Secondary transformer"),
    mlines.Line2D([0], [0], marker="s", color="#f39c12", markersize=10,
                   linestyle="None", label="Assigned transformer"),
    mpatches.Patch(facecolor="#aed6f1", edgecolor="#5d6d7e", label="Voronoi cell"),
]
ax.legend(handles=legend_handles, fontsize=9, loc="upper right")

plt.tight_layout()
plt.savefig("../data/outputs/04_voronoi_assignment_map.png", dpi=150, bbox_inches="tight")
plt.show()

## 4.5  Full Grid Path Lookup

In [ ]:
# Run the full PathBuilder pipeline
path_builder = PathBuilder(
    G=G,
    ldc_resolver=ldc_resolver,
    transformer_finder=TransformerFinder(G),
    geocoder=geocoder,
)

result = path_builder.lookup_address(DEMO_ADDRESS, radius_m=300.0)

print(f"\nADDRESS LOOKUP RESULT")
print("=" * 55)
for key, val in result.items():
    if key == "path_nodes":
        print(f"  {'path_nodes':<28} ({len(val)} nodes)")
    else:
        print(f"  {key:<28} {val}")
print("=" * 55)

## 4.6  Grid Path Visualisation — Building to Generator

In [ ]:
path_nodes = result.get("path_nodes", [])

if len(path_nodes) < 2:
    print("Path too short to visualise — no full path found in graph.")
else:
    # Build a sub-graph of the path for drawing
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.set_facecolor("#f8f9fa")

    node_type_colors = {
        "generator":              "#f39c12",
        "tx_substation":          "#c0392b",
        "zone_substation":        "#e74c3c",
        "dx_substation":          "#e67e22",
        "dx_feeder_node":         "#3498db",
        "secondary_transformer":  "#27ae60",
        "building":               "#95a5a6",
    }
    node_type_sizes = {
        "generator":              800,
        "tx_substation":          600,
        "zone_substation":        500,
        "dx_substation":          400,
        "dx_feeder_node":         250,
        "secondary_transformer":  200,
        "building":               150,
    }

    # Lay out the path as a horizontal chain
    n = len(path_nodes)
    x_positions = np.linspace(0, n - 1, n)
    y_positions = np.zeros(n)

    # Add small vertical jitter for readability
    rng = np.random.default_rng(seed=0)
    y_positions += rng.normal(0, 0.05, n)

    pos = {node: (x_positions[i], y_positions[i]) for i, node in enumerate(path_nodes)}

    # Draw edges
    for i in range(n - 1):
        u, v = path_nodes[i], path_nodes[i + 1]
        xu, yu = pos[u]
        xv, yv = pos[v]
        edge_data = G.edges.get((u, v), {})
        etype = edge_data.get("edge_type", "")
        econf = edge_data.get("confidence", 0.5)
        ecolor = {"tx_line": "#c0392b", "primary_feeder": "#e67e22",
                  "secondary_lateral": "#27ae60", "service_drop": "#95a5a6"}.get(etype, "#bdc3c7")
        lw = 1.5 + 2 * econf
        ax.annotate(
            "", xy=(xv, yv), xytext=(xu, yu),
            arrowprops=dict(arrowstyle="->", color=ecolor, lw=lw)
        )
        # Label edge type
        mid_x = (xu + xv) / 2
        mid_y = (yu + yv) / 2 + 0.12
        ax.text(mid_x, mid_y, f"{etype}\nconf={econf:.2f}",
                fontsize=6, ha="center", color="#5d6d7e")

    # Draw nodes
    for i, node in enumerate(path_nodes):
        ndata = G.nodes.get(node, {})
        ntype = ndata.get("node_type", "unknown")
        nconf = ndata.get("confidence", 0.5)
        color = node_type_colors.get(ntype, "#bdc3c7")
        size = node_type_sizes.get(ntype, 200)
        x, y = pos[node]
        ax.scatter(x, y, s=size, c=color, zorder=5, edgecolors="white", linewidths=1.5)
        name = ndata.get("name", ntype)
        label_str = f"{ntype}\n{name[:20]}\nconf={nconf:.2f}"
        ax.text(x, y - 0.25, label_str, ha="center", va="top",
                fontsize=7, color="#2c3e50",
                bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7, edgecolor="none"))

    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(-0.6, 0.6)
    ax.set_title(
        f"Full Grid Path: Building \u2192 Generator\n"
        f"{DEMO_ADDRESS}\n"
        f"Path length: {len(path_nodes)} nodes | Min confidence: {result.get('confidence_min', 0):.2f} "
        f"({result.get('confidence_label', '')})",
        fontsize=11, fontweight="bold"
    )
    ax.axis("off")

    legend_handles = [
        mpatches.Patch(color=v, label=k)
        for k, v in node_type_colors.items()
        if any(G.nodes.get(n, {}).get("node_type") == k for n in path_nodes)
    ]
    ax.legend(handles=legend_handles, fontsize=8, loc="upper left",
              title="Node type", title_fontsize=8)

    plt.tight_layout()
    plt.savefig("../data/outputs/04_grid_path_building_to_generator.png",
                dpi=150, bbox_inches="tight")
    plt.show()

## 4.7  Result Summary

In [ ]:
print("=" * 60)
print("  SERVICE POINT LOOKUP — FINAL SUMMARY")
print("=" * 60)
print(f"  Address:                {DEMO_ADDRESS}")
print(f"  Status:                 {result.get('status')}")
print(f"  Geocoded lat/lon:       {result.get('geocoded_lat'):.6f}, {result.get('geocoded_lon'):.6f}")
print(f"  LDC:                    {result.get('ldc_name')} (source: {result.get('ldc_source')})")
print(f"  Secondary transformer:  {result.get('secondary_transformer')}")
print(f"    distance:             {result.get('tx_distance_m'):.1f} m")
print(f"  DX substation:          {result.get('dx_substation')}")
print(f"  Zone substation:        {result.get('zone_substation')}")
print(f"  TX substation:          {result.get('tx_substation')}")
print(f"  Generator:              {result.get('generator')}")
print(f"  Path length (nodes):    {result.get('path_length')}")
print(f"  Min confidence:         {result.get('confidence_min'):.2f}  ({result.get('confidence_label')})")
print(f"  SAR flood exposure:     {result.get('sar_flood_exposure')}")
print(f"  SAR risk label:         {result.get('sar_risk_label')}")
print("=" * 60)